# Pure-State Entanglement Slope vs System Size

This notebook reads completed `pure_state_entanglement_slope_vs_system_size_time_scaling` runs and plots fitted log-chord slope with fit-derived error bars, plus `1 - R^2` on a log y-scale, versus system size.

## 1. Setup

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    BUNDLE_ROOT = Path('/content/drive/MyDrive/colab_large_entanglement_scaling_N20')
else:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    BUNDLE_ROOT = next(
        (path / 'colab_large_entanglement_scaling_N20' for path in candidates if (path / 'colab_large_entanglement_scaling_N20').exists()),
        cwd.parent if cwd.name == 'notebooks' else cwd,
    )

DATASET_NAME = 'pure_state_entanglement_slope_vs_system_size_time_scaling'
DATA_ROOT = BUNDLE_ROOT / 'gpu_data' / DATASET_NAME
RUNS_ROOT = DATA_ROOT / 'runs'
FIG_ROOT = DATA_ROOT / 'figures'
FIG_ROOT.mkdir(parents=True, exist_ok=True)

print(f'[setup] bundle root: {BUNDLE_ROOT}')
print(f'[setup] data root: {DATA_ROOT}')
print(f'[setup] figures root: {FIG_ROOT}')

## 2. Load Completed Fits

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

if not RUNS_ROOT.exists():
    raise FileNotFoundError(RUNS_ROOT)

rows: list[dict[str, object]] = []

for run_dir in sorted(RUNS_ROOT.iterdir()):
    if not run_dir.is_dir():
        continue

    summary_path = run_dir / 'run_summary.json'
    fit_rows_path = run_dir / 'fit_rows.csv'

    if summary_path.exists():
        summary = json.loads(summary_path.read_text())
        if summary.get('status') != 'complete':
            continue

        cfg = summary.get('config', {})
        fit = summary.get('fit', {})
        rows.append({
            'run_id': summary.get('run_id', run_dir.name),
            'Nx': cfg.get('Nx', fit.get('Nx')),
            'Ny': cfg.get('Ny', fit.get('Ny')),
            'Nsites': int(cfg.get('Nx', fit.get('Nx'))) * int(cfg.get('Ny', fit.get('Ny'))),
            'cycles': cfg.get('cycles', fit.get('cycle')),
            'samples': cfg.get('samples'),
            'slope': fit.get('slope'),
            'slope_err': fit.get('slope_err'),
            'target_slope': fit.get('target_slope'),
            'r2': fit.get('r2'),
            'percent_diff': fit.get('percent_diff'),
            'n_fit_points': fit.get('n_fit_points'),
            'source': str(summary_path.relative_to(DATA_ROOT)),
        })
        continue

    if fit_rows_path.exists():
        fit_df = pd.read_csv(fit_rows_path)
        if fit_df.empty:
            continue
        row = fit_df.iloc[0].to_dict()
        nx = int(row['Nx'])
        ny = int(row['Ny'])
        rows.append({
            'run_id': run_dir.name,
            'Nx': nx,
            'Ny': ny,
            'Nsites': nx * ny,
            'cycles': row.get('cycle'),
            'samples': np.nan,
            'slope': row.get('slope'),
            'slope_err': row.get('slope_err'),
            'target_slope': row.get('target_slope'),
            'r2': row.get('r2'),
            'percent_diff': row.get('percent_diff'),
            'n_fit_points': row.get('n_fit_points'),
            'source': str(fit_rows_path.relative_to(DATA_ROOT)),
        })

df = pd.DataFrame(rows)
if df.empty:
    raise RuntimeError(f'No completed fits found under {RUNS_ROOT}')

numeric_cols = ['Nx', 'Ny', 'Nsites', 'cycles', 'samples', 'slope', 'slope_err', 'target_slope', 'r2', 'percent_diff', 'n_fit_points']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df = df.sort_values(['Ny', 'Nx']).reset_index(drop=True)
required = {'Ny', 'Nsites', 'slope', 'slope_err', 'r2'}
missing_values = [col for col in required if df[col].isna().any()]
if missing_values:
    raise ValueError(f'Completed fit rows have missing values in required columns: {missing_values}')

display(Markdown(f'### Loaded {len(df)} completed fits'))
display(df[['run_id', 'Nx', 'Ny', 'Nsites', 'cycles', 'samples', 'slope', 'slope_err', 'r2', 'percent_diff', 'n_fit_points']])

## 3. Plot Slope and 1 - R2

In [ ]:
# Use Ny as the default system-size coordinate because these runs keep Nx fixed at 20.
# Change this to 'Nsites' to plot against total lattice sites instead.
X_COLUMN = 'Ny'
X_LABEL = 'system size Ny (Nx = 20)'

x = df[X_COLUMN].to_numpy(dtype=np.float64)
slope = df['slope'].to_numpy(dtype=np.float64)
slope_err = df['slope_err'].to_numpy(dtype=np.float64)
r2 = df['r2'].to_numpy(dtype=np.float64)
one_minus_r2 = 1.0 - r2
positive_residual = np.isfinite(one_minus_r2) & (one_minus_r2 > 0.0)
if not np.all(positive_residual):
    bad_rows = df.loc[~positive_residual, ['run_id', 'Ny', 'r2']]
    raise ValueError(f'Cannot use log y-scale because some 1 - R^2 values are non-positive or non-finite:\n{bad_rows}')
target_slope_values = df['target_slope'].dropna().unique()
target_slope = float(target_slope_values[0]) if len(target_slope_values) else 1.0 / 3.0

slope_path = FIG_ROOT / f'slope_vs_{X_COLUMN}_with_fit_errorbars.png'
one_minus_r2_path = FIG_ROOT / f'one_minus_r2_vs_{X_COLUMN}_logy.png'
combined_path = FIG_ROOT / f'slope_and_one_minus_r2_vs_{X_COLUMN}.png'

fig, ax = plt.subplots(figsize=(7.4, 4.8), constrained_layout=True)
ax.errorbar(x, slope, yerr=slope_err, marker='o', ms=5, lw=1.4, capsize=3, label='fit slope')
ax.axhline(target_slope, color='black', ls='--', lw=1.0, label='target 1/3')
ax.set_xlabel(X_LABEL)
ax.set_ylabel('full-x log-chord slope')
ax.set_title('Pure-state entanglement slope vs system size')
ax.grid(True, alpha=0.25)
ax.legend(frameon=True)
fig.savefig(slope_path, dpi=200)
plt.show()

fig, ax = plt.subplots(figsize=(7.4, 4.8), constrained_layout=True)
ax.plot(x, one_minus_r2, marker='o', ms=5, lw=1.4)
ax.set_yscale('log')
ax.set_xlabel(X_LABEL)
ax.set_ylabel(r'$1 - R^2$')
ax.set_title(r'Fit residual $1 - R^2$ vs system size')
ax.grid(True, which='both', alpha=0.25)
fig.savefig(one_minus_r2_path, dpi=200)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13.0, 4.8), constrained_layout=True)
axes[0].errorbar(x, slope, yerr=slope_err, marker='o', ms=5, lw=1.4, capsize=3, label='fit slope')
axes[0].axhline(target_slope, color='black', ls='--', lw=1.0, label='target 1/3')
axes[0].set_xlabel(X_LABEL)
axes[0].set_ylabel('full-x log-chord slope')
axes[0].set_title('slope')
axes[0].grid(True, alpha=0.25)
axes[0].legend(frameon=True)

axes[1].plot(x, one_minus_r2, marker='o', ms=5, lw=1.4)
axes[1].set_yscale('log')
axes[1].set_xlabel(X_LABEL)
axes[1].set_ylabel(r'$1 - R^2$')
axes[1].set_title(r'$1 - R^2$')
axes[1].grid(True, which='both', alpha=0.25)

fig.suptitle('Pure-state entanglement fit quality vs system size')
fig.savefig(combined_path, dpi=200)
plt.show()

print(f'[saved] {slope_path}')
print(f'[saved] {one_minus_r2_path}')
print(f'[saved] {combined_path}')

## 4. Saved Figures

In [ ]:
display(Markdown('### Slope vs system size'))
display(Image(filename=str(slope_path)))
display(Markdown(r'### $1 - R^2$ vs system size, log y'))
display(Image(filename=str(one_minus_r2_path)))
display(Markdown('### Combined'))
display(Image(filename=str(combined_path)))

## 5. Disconnect Runtime

In [ ]:
if IN_COLAB:
    from google.colab import runtime  # type: ignore
    runtime.unassign()
else:
    print('[done] not running in Colab; runtime disconnect skipped')